In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/blake3-1.0.8-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/shellingham-1.5.4-py2.py3-none-any.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/vllm-wheels/typing_extensions-4.15.0-py3-none-any.whl
/kaggle/input/datasets/liquidvisualsinteractive/vllm-wheels/v

In [2]:
%%writefile test.py
import os
import random
import sys

# --- CONFIGURATION ---
SET_SIZE = 50 # Default for non-interactive benchmark

# --- 🧠 Problem Bank (Tiers) ---
EASY_POOL = [
    {"id": "01", "problem": "Find the sum of all prime numbers $p$ such that $p+2$ and $p+4$ are also prime.", "answer": 3},
    {"id": "02", "problem": "Let $P(x) = x^2 - 10x + 25$. Find the value of $P(15)$.", "answer": 100},
    {"id": "03", "problem": "Find the number of positive integers $n < 100$ such that $n$ is a multiple of 3 and $n+1$ is a multiple of 4.", "answer": 9},
    {"id": "04", "problem": "If $a$ and $b$ are roots of $x^2 - 14x + 48 = 0$, find $a^2 + b^2$.", "answer": 100},
    {"id": "05", "problem": "Find the sum of the digits of the integer $10^{10} - 1$.", "answer": 90},
    {"id": "06", "problem": "A sequence $a_n$ is defined by $a_1 = 3$ and $a_{n+1} = a_n^2 - 2$. Find $a_3$.", "answer": 47},
    {"id": "07", "problem": "Find the number of ordered pairs of integers $(x,y)$ such that $x^2 + y^2 = 25$.", "answer": 12},
    {"id": "08", "problem": "Find the sum of all real roots of $(x-1)(x-2)(x-3) = 0$.", "answer": 6},
    {"id": "09", "problem": "Evaluate the infinite sum $\\sum_{n=1}^\\infty \\frac{n}{3^n}$. Express your answer as $m/n$, and find $10m+n$.", "answer": 34},
    {"id": "10", "problem": "Find the smallest positive integer $n$ such that $n!$ is divisible by 1000.", "answer": 15},
    {"id": "11", "problem": "Find the number of integers $x \\in \\{1, 2, \\dots, 1000\\}$ that are relatively prime to 10.", "answer": 400},
    {"id": "12", "problem": "Find the constant term in the expansion of $(x + \\frac{2}{x})^6$.", "answer": 160},
    {"id": "13", "problem": "Find the number of three-digit integers $abc$ such that $a < b < c$.", "answer": 84},
    {"id": "14", "problem": "Find the smallest positive integer $n$ such that $2^n \\equiv 1 \\pmod{31}$.", "answer": 5},
    {"id": "15", "problem": "Evaluate \\sqrt{12 + \\sqrt{12 + \\sqrt{12 + \\dots}}}.", "answer": 4},
    {"id": "16", "problem": "Find the sum of the first 100 terms of the arithmetic progression $5, 8, 11, \\dots$.", "answer": 15350},
    {"id": "17", "problem": "Find the number of subsets of \\{1, 2, 3, 4, 5, 6, 7, 8\\} that do not contain two consecutive integers.", "answer": 55},
    {"id": "18", "problem": "Find the sum of the squares of the coefficients of $(x+1)^4$.", "answer": 70},
    {"id": "19", "problem": "Find the number of positive integers $n$ for which $n^2 + 8n + 15$ is prime.", "answer": 0},
    {"id": "20", "problem": "Find the remainder when $3^{2024}$ is divided by 100.", "answer": 81},
    {"id": "21", "problem": "Evaluate $100^2 - 99^2 + 98^2 - 97^2 + \\dots + 2^2 - 1^2$.", "answer": 5050},
    {"id": "22", "problem": "Find the value of $n$ such that \\binom{n}{2} = 4950.", "answer": 100},
    {"id": "23", "problem": "Find the number of ways to arrange the letters in the word 'KAGGLE'.", "answer": 360},
    {"id": "24", "problem": "A circle is inscribed in a right triangle with sides 3, 4, and 5. Find the square of the area of the circle multiplied by $(100/\\pi)$.", "answer": 314},
    {"id": "25", "problem": "Find the sum of the zeros of $P(x) = x^3 - 6x^2 + 11x - 6$.", "answer": 6},
    {"id": "26", "problem": "Find the smallest integer $n > 1$ such that $n^2$ is a cube and $n^3$ is a square.", "answer": 64},
    {"id": "27", "problem": "Find the number of divisors of $2^5 \\cdot 3^4 \\cdot 5^3$.", "answer": 120},
    {"id": "28", "problem": "Find the sum of the interior angles (in degrees) of a convex decagon.", "answer": 1440},
    {"id": "29", "problem": "How many positive integers $n < 500$ are not divisible by 2, 3, or 5?", "answer": 134},
    {"id": "30", "problem": "Find the value of $x$ satisfying \\log_2(\\log_3(\\log_4 x)) = 0.", "answer": 64},
    {"id": "31", "problem": "A square has a perimeter of 40. Find its area.", "answer": 100},
    {"id": "32", "problem": "Find the hypotenuse of a right triangle with legs 8 and 15.", "answer": 17},
    {"id": "33", "problem": "Find the sum of the interior angles of a hexagon in degrees.", "answer": 720},
    {"id": "34", "problem": "A circle has area $49\\pi$. Find its diameter.", "answer": 14},
    {"id": "35", "problem": "Find the area of a triangle with base 10 and height 12.", "answer": 60},
    {"id": "36", "problem": "How many ways can 3 people be seated in 3 chairs?", "answer": 6},
    {"id": "37", "problem": "Find the number of subsets of {1, 2, 3, 4}.", "answer": 16},
    {"id": "38", "problem": "How many ways to choose 2 colors from 5 distinct colors?", "answer": 10},
    {"id": "39", "problem": "A bag has 3 red and 2 blue marbles. How many ways to pick 2 marbles (order matters)?", "answer": 20},
    {"id": "40", "problem": "Find the number of permutations of the letters in 'MATH'.", "answer": 24},
    {"id": "41", "problem": "A fair coin is flipped twice. Find the probability of 2 heads. Multiply by 100.", "answer": 25},
    {"id": "42", "problem": "A die is rolled. Find the probability of rolling a 1 or 2. Multiply by 60.", "answer": 20},
    {"id": "43", "problem": "If the probability of rain is 0.3, find the prob. it doesn't rain. Multiply by 100.", "answer": 70},
    {"id": "44", "problem": "A bag has 10 balls. 3 are red. Prob of picking a red? Multiply by 100.", "answer": 30},
    {"id": "45", "problem": "Expected value of a die roll is 3.5. Multiply by 10.", "answer": 35},
    {"id": "46", "problem": "Find the 5th term of the sequence 2, 5, 8, 11...", "answer": 14},
    {"id": "47", "problem": "Find the sum of 1, 2, 4, 8, 16.", "answer": 31},
    {"id": "48", "problem": "Find the common ratio of the geometric sequence 3, 6, 12, 24.", "answer": 2},
    {"id": "49", "problem": "Find the 10th term of $a_n = 2n + 1$.", "answer": 21},
    {"id": "50", "problem": "Sum of first 10 positive integers.", "answer": 55},
    {"id": "51", "problem": "Find the greatest common divisor of 12 and 18.", "answer": 6},
    {"id": "52", "problem": "Find the remainder when 100 is divided by 7.", "answer": 2},
    {"id": "53", "problem": "Find the smallest prime number greater than 10.", "answer": 11},
    {"id": "54", "problem": "Find the number of divisors of 12.", "answer": 6},
    {"id": "55", "problem": "What is the 5th prime number?", "answer": 11},
    {"id": "56", "problem": "Solve for x: 3x + 5 = 20.", "answer": 5},
    {"id": "57", "problem": "If f(x) = x^2 + 1, find f(3).", "answer": 10},
    {"id": "58", "problem": "Factor x^2 - 9. Find the positive root.", "answer": 3},
    {"id": "59", "problem": "Find the slope of the line y = 4x + 7.", "answer": 4},
    {"id": "60", "problem": "Evaluate (2+3)^2 - 5.", "answer": 20},
    {"id": "61", "problem": "Find the smallest integer x such that x > 5.5.", "answer": 6},
    {"id": "62", "problem": "Find the maximum value of 10 - x^2.", "answer": 10},
    {"id": "63", "problem": "If x < 5 and x is an integer, find the largest x.", "answer": 4},
    {"id": "64", "problem": "Solve for x: |x| = 7 and x < 0. Express |x|.", "answer": 7},
    {"id": "65", "problem": "Is 3 > 2? (1 for Yes, 0 for No).", "answer": 1},
    {"id": "66", "problem": "If Alice starts with 10 apples and gives 3 to Bob, how many left?", "answer": 7},
    {"id": "67", "problem": "In a game, the winner gets 5 points. If Bob wins 3 times, how many points?", "answer": 15},
    {"id": "68", "problem": "A game has 2 players. If Alice wins, Bob loses. Result is 1. If tie 0. If Alice wins, result?", "answer": 1},
    {"id": "69", "problem": "Two people flip a coin. Alice wins on Heads. Heads occurs. Who wins? (1 for Alice, 2 for Bob).", "answer": 1},
    {"id": "70", "problem": "If a strategy guarantees a win in 5 moves, how many moves to win?", "answer": 5},
    {"id": "71", "problem": "If f(x) = 2x, find f(10).", "answer": 20},
    {"id": "72", "problem": "If f(x) = f(x+1) - 1 and f(0) = 0, find f(5).", "answer": 5},
    {"id": "73", "problem": "If f(f(x)) = x and f(1) = 2, find f(2).", "answer": 1},
    {"id": "74", "problem": "A graph has 4 vertices and 0 edges. Find the number of vertices.", "answer": 4},
    {"id": "75", "problem": "Find the number of edges in a triangle graph.", "answer": 3}
]

MEDIUM_POOL = [
    {"id": "76", "problem": "Find the coefficient of $x^7$ in $(1+x)^{10}$.", "answer": 120},
    {"id": "77", "problem": "Find the number of solutions to $x_1 + x_2 + x_3 = 10$ in non-negative integers.", "answer": 66},
    {"id": "78", "problem": "Evaluate \\sum_{k=1}^{10} (k^3 - k).", "answer": 2970},
    {"id": "79", "problem": "Find the number of trailing zeros in $100!$.", "answer": 24},
    {"id": "80", "problem": "Find the sum of all three-digit palindromes.", "answer": 49500},
    {"id": "81", "problem": "If \\tan x = 3/4, find the value of $100 \\sin(2x)$.", "answer": 96},
    {"id": "82", "problem": "Find the smallest positive integer $n$ such that $2^n > n^{10}$.", "answer": 59},
    {"id": "83", "problem": "Find the number of positive integers $n \\le 1000$ such that \\lfloor \\sqrt{n} \\rfloor divides $n$.", "answer": 92},
    {"id": "84", "problem": "Find the value of $1 \\cdot 1! + 2 \\cdot 2! + \\dots + 6 \\cdot 6!$.", "answer": 5039},
    {"id": "85", "problem": "Find the product of the roots of $x^4 - 5x^2 + 4 = 0$.", "answer": 4},
    {"id": "86", "problem": "How many ways can 6 people be seated around a circular table (reflections distinct)?", "answer": 120},
    {"id": "87", "problem": "Find the sum of the first 50 odd numbers.", "answer": 2500},
    {"id": "88", "problem": "Find the value of $1234^2 - 1233 \\cdot 1235$.", "answer": 1},
    {"id": "89", "problem": "Find the number of integer points $(x,y)$ inside or on the circle $x^2 + y^2 = 10$.", "answer": 37},
    {"id": "90", "problem": "Find the last digit of $7^{7^7}$.", "answer": 3},
    {"id": "91", "problem": "A fair coin is flipped 10 times. Find the number of outcomes with exactly 5 heads.", "answer": 252},
    {"id": "92", "problem": "Find the product of all distinct prime factors of 2024.", "answer": 506},
    {"id": "93", "problem": "Find the value of $x$ if $x + \\frac{1}{x} = 3$, find $x^4 + \\frac{1}{x^4}$.", "answer": 47},
    {"id": "94", "problem": "Find the sum of all integers from 1 to 446.", "answer": 99681},
    {"id": "95", "problem": "Find the largest 5-digit palindrome divisible by 9.", "answer": 99999},
    {"id": "96", "problem": "Find the number of integers $n$ such that $n+3$ divides $n^2+7$.", "answer": 10},
    {"id": "97", "problem": "Find the sum of all real roots of $x^2 - 100x + 99 = 0$.", "answer": 100},
    {"id": "98", "problem": "Find the number of subsets of \\{1, 2, \\dots, 10\\} with exactly 3 elements such that no two elements are consecutive.", "answer": 56},
    {"id": "99", "problem": "Find the remainder when $2^{2025}$ is divided by 13.", "answer": 5},
    {"id": "100", "problem": "Find the value of \\sum_{k=1}^{100} \\lfloor \\log_2 k \\rfloor.", "answer": 480},
    {"id": "101", "problem": "How many positive integers divide at least two of the integers 60, 126, and 140?", "answer": 10},
    {"id": "102", "problem": "Find the number of ordered triples $(a,b,c)$ of positive integers such that $a+b+c = 15$.", "answer": 91},
    {"id": "103", "problem": "Find the sum of all digits of $11^{11}$.", "answer": 41},
    {"id": "104", "problem": "Find the absolute difference between the roots of $x^2 - 100x + 99 = 0$.", "answer": 98},
    {"id": "105", "problem": "Find the number of solutions to $x^2 \\equiv 1 \\pmod{100}$.", "answer": 4},
    {"id": "106", "problem": "Find the area of a triangle with side lengths 13, 14, and 15.", "answer": 84},
    {"id": "107", "problem": "Evaluate \\int_0^2 (x^3 - 3x^2 + 4) dx. Multiply by 10.", "answer": 40},
    {"id": "108", "problem": "Find the smallest $n$ such that $1+2+\\dots+n > 1000$.", "answer": 45},
    {"id": "109", "problem": "Find the number of ways to color the edges of a square with 3 colors s.t. adjacent edges have different colors.", "answer": 18},
    {"id": "110", "problem": "Find the largest prime factor of $3^{10} - 1$.", "answer": 61},
    {"id": "111", "problem": "Find the value of $x^3+y^3$ if $x+y=5$ and $xy=6$.", "answer": 35},
    {"id": "112", "problem": "Find the number of 4-digit numbers where all digits are different and odd.", "answer": 120},
    {"id": "113", "problem": "Find the minimum value of $x^2 - 12x + 40$.", "answer": 4},
    {"id": "114", "problem": "Find the sum of all $n$ such that $n$ is a divisor of $100$ but not a divisor of $50$.", "answer": 124},
    {"id": "115", "problem": "Find the number of ways to stack 4 identical blocks in 3 different bins.", "answer": 15},
    {"id": "116", "problem": "A right triangle has legs of length 10 and 24. Find the radius of its circumcircle.", "answer": 13},
    {"id": "117", "problem": "Find the length of the diagonal of a rectangular box with dimensions 1, 2, and 2.", "answer": 3},
    {"id": "118", "problem": "A circle is tangent to both axes in the first quadrant and passes through (2, 4). Find the larger possible radius.", "answer": 10},
    {"id": "119", "problem": "Find the sum of the first 15 terms of the arithmetic sequence 2, 9, 16, 23, ...", "answer": 765},
    {"id": "120", "problem": "If $\\sin x + \\cos x = 1.2$, find $100 \\sin(2x)$.", "answer": 44},
    {"id": "121", "problem": "Find the volume of a sphere with radius 3. Multiply by $(3/\\pi)$.", "answer": 108},
    {"id": "122", "problem": "Find the number of ways to choose 4 items from 10.", "answer": 210},
    {"id": "123", "problem": "Find the number of anagrams of 'BANANA'.", "answer": 60},
    {"id": "124", "problem": "How many ways to partition 6 items into two groups of 3?", "answer": 10},
    {"id": "125", "problem": "Find the number of integer solutions to x+y+z=10 with x,y,z > 0.", "answer": 36},
    {"id": "126", "problem": "A grid of 3x3 dots. How many squares can be formed?", "answer": 6},
    {"id": "127", "problem": "How many ways to seat 5 people in a line if Alice and Bob must be adjacent?", "answer": 48},
    {"id": "128", "problem": "A fair coin is flipped 3 times. What is the expected number of heads? Multiply by 10.", "answer": 15},
    {"id": "129", "problem": "A die is rolled until a 6 appears. Expected number of rolls?", "answer": 6},
    {"id": "130", "problem": "A bag has 5 red and 5 blue balls. If you draw 2, expected red? Multiply by 10.", "answer": 10},
    {"id": "131", "problem": "Two dice are rolled. Sum is X. Expected value of X?", "answer": 7},
    {"id": "132", "problem": "A coin is flipped. Heads you get 10 points, Tails 0. Expected points?", "answer": 5},
    {"id": "133", "problem": "Probability of rain is 1/2. Expected days of rain in a week? Multiply by 2.", "answer": 7},
    {"id": "134", "problem": "Find the sum of the first 20 terms of $a_n = n^2$.", "answer": 2870},
    {"id": "135", "problem": "Find the limit of $n/(2n+1)$ as n goes to infinity. Multiply by 10.", "answer": 5},
    {"id": "136", "problem": "Find the 10th Fibonacci number starting from $F_1=1, F_2=1$.", "answer": 55},
    {"id": "137", "problem": "Find the sum of the geometric series $1 + 1/2 + 1/4 + \\dots$. Multiply by 10.", "answer": 20},
    {"id": "138", "problem": "Sequence $a_1=1, a_n = 2a_{n-1} + 1$. Find $a_6$.", "answer": 63},
    {"id": "139", "problem": "Find the product of the first 5 terms of $a_n = 2^n$.", "answer": 32768},
    {"id": "140", "problem": "Find the smallest $n$ such that $n \\equiv 1 \\pmod 3$ and $n \\equiv 2 \\pmod 5$.", "answer": 7},
    {"id": "141", "problem": "Find the remainder of $123456 \\pmod{11}$.", "answer": 3},
    {"id": "142", "problem": "Find the number of digits in $2^{10} \\cdot 5^7$.", "answer": 8},
    {"id": "143", "problem": "Find the largest prime factor of 1001.", "answer": 13},
    {"id": "144", "problem": "Find the number of positive integers $n < 100$ divisible by both 4 and 6.", "answer": 8},
    {"id": "145", "problem": "If $x$ is an integer and $x^2 \\equiv 4 \\pmod 7$, find the smallest positive $x$.", "answer": 2},
    {"id": "146", "problem": "Find the sum of the roots of $2x^2 - 100x + 1 = 0$.", "answer": 50},
    {"id": "147", "problem": "If $P(x) = x^3 - 12x^2 + 47x - 60$, find the product of its roots.", "answer": 60},
    {"id": "148", "problem": "Solve for x: $x + 2\\sqrt{x} = 15$.", "answer": 9},
    {"id": "149", "problem": "Find the value of $k$ if $x=2$ is a root of $x^2 + kx - 10 = 0$.", "answer": 3},
    {"id": "150", "problem": "Evaluate $(1+i)^8$.", "answer": 16},
    {"id": "151", "problem": "Find the vertex of $y = x^2 - 4x + 7$. Express the y-coordinate.", "answer": 3},
    {"id": "152", "problem": "Find the minimum value of $x + 1/x$ for $x > 0$.", "answer": 2},
    {"id": "153", "problem": "If $x+y=10$, find the maximum of $xy$.", "answer": 25},
    {"id": "154", "problem": "Find the smallest $x$ such that $2x^2 - 8x + 10 \\le 2$.", "answer": 2},
    {"id": "155", "problem": "If $a, b > 0$ and $a+b=6$, find max of $a^2b$.", "answer": 32},
    {"id": "156", "problem": "Find the sum of all integers x such that $x^2 - 5x + 6 \\le 0$.", "answer": 5},
    {"id": "157", "problem": "Find the minimum of $e^x + e^{-x}$. Round to nearest integer.", "answer": 2},
    {"id": "158", "problem": "Alice and Bob play a game. Alice starts with 10. Each turn they subtract 1. Last to 0 wins. Who wins? (1-A, 2-B).", "answer": 2},
    {"id": "159", "problem": "In a game of 21, the sum starts at 0. Players add 1, 2, or 3. Player who makes it 21 wins. First player adds X. Find X to win.", "answer": 1},
    {"id": "160", "problem": "If $x$ and $y$ are positive integers such that $x^2 - y^2 = 51$, find the maximum possible value of $x+y$.", "answer": 51},
    {"id": "161", "problem": "Tic-Tac-Toe has how many squares?", "answer": 9},
    {"id": "162", "problem": "Alice wins if she flips Heads. Prob is 0.6. Expected wins in 100 flips?", "answer": 60},
    {"id": "163", "problem": "Total number of states in a 2-node game where each node can be 0 or 1.", "answer": 4},
    {"id": "164", "problem": "If f(x+y) = f(x) + f(y) and f(1) = 3, find f(10).", "answer": 30},
    {"id": "165", "problem": "If f(x) = ax + b, f(1)=5, f(2)=7, find f(10).", "answer": 23},
    {"id": "166", "problem": "If f(x) = f(x-1) + 2x - 1 and f(0) = 0, find f(4).", "answer": 16},
    {"id": "167", "problem": "If f(f(x)) = x+2 and f(0)=1, find f(1).", "answer": 2},
    {"id": "168", "problem": "If f(x^2) = x^4 + 1, find f(5).", "answer": 26},
    {"id": "169", "problem": "Evaluate the sum of the coefficients of the polynomial $P(x) = (3x^2 - 2x + 1)^4$.", "answer": 16},
    {"id": "170", "problem": "A graph has 10 vertices. Total possible edges?", "answer": 45},
    {"id": "171", "problem": "Find the degree of a vertex in a cycle graph $C_5$.", "answer": 2},
    {"id": "172", "problem": "Number of edges in a tree with 100 vertices.", "answer": 99},
    {"id": "173", "problem": "Maximum number of edges in a bipartite graph with 2+2 vertices.", "answer": 4},
    {"id": "174", "problem": "How many colors needed for a bipartite graph?", "answer": 2},
    {"id": "175", "problem": "Number of vertices in a cube.", "answer": 8}
]

HARD_POOL = [
    {"id": "176", "problem": "Find the value of \\sin(15^\\circ) \\cos(15^\\circ) multiplied by 400.", "answer": 100},
    {"id": "177", "problem": "Find the largest value of $x$ for which \\sqrt{x} + \\sqrt{x-5} = 5.", "answer": 9},
    {"id": "178", "problem": "Find the number of permutations of (1, 2, 3, 4, 5, 6) with no fixed points.", "answer": 265},
    {"id": "179", "problem": "Evaluate $2^{2^3} - (2^2)^3$.", "answer": 192},
    {"id": "180", "problem": "Find the radius of the incircle of a 3-4-5 triangle.", "answer": 1},
    {"id": "181", "problem": "Find the number of real solutions to $x^4 - 4x^2 + 4 = 0$.", "answer": 2},
    {"id": "182", "problem": "Find the greatest common divisor of $2^{10}-1$ and $2^{15}-1$.", "answer": 31},
    {"id": "183", "problem": "Find the sum of all integers $x$ such that |x-5| < 3.", "answer": 25},
    {"id": "184", "problem": "Find the value of $n$ if $1+3+5+\\dots+(2n-1) = 10000$.", "answer": 100},
    {"id": "185", "problem": "Find the units digit of $2^{2024}$.", "answer": 6},
    {"id": "186", "problem": "How many ways can 10 distinct books be given to 2 children such that each child gets 5?", "answer": 252},
    {"id": "187", "problem": "Find the number of three-digit numbers divisible by 7.", "answer": 128},
    {"id": "188", "problem": "Find the sum of the roots of $x^2 - 2024x + 2023 = 0$.", "answer": 2024},
    {"id": "189", "problem": "Find the remainder when $x^{100}$ is divided by $x-1$.", "answer": 1},
    {"id": "190", "problem": "Find the area of a circle with circumference $20\\pi$.", "answer": 314},
    {"id": "191", "problem": "Find the number of integers $n$ such that $n^2+1$ is prime and $n \\le 10$.", "answer": 5},
    {"id": "192", "problem": "Find the value of $1/2 + 1/4 + 1/8 + \\dots$ up to 10 terms, then multiply by 1024 and subtract from 1024.", "answer": 1},
    {"id": "193", "problem": "Find the sum of the digits of $123 \\times 456$.", "answer": 27},
    {"id": "194", "problem": "Find the number of ways to choose 2 numbers from \\{1, 2, \\dots, 20\\} whose sum is even.", "answer": 90},
    {"id": "195", "problem": "Find the smallest prime $p > 100$.", "answer": 101},
    {"id": "196", "problem": "Find the number of edges in a complete graph with 10 vertices.", "answer": 45},
    {"id": "197", "problem": "Evaluate $3^5 - 5^3$.", "answer": 118},
    {"id": "198", "problem": "Find the number of positive integers $k < 20$ such that $x^2 + kx + 16 = 0$ has real roots.", "answer": 12}, 
    {"id": "199", "problem": "Find the value of $111 \\times 111$.", "answer": 12321},
    {"id": "200", "problem": "Find the number of paths from (0,0) to (3,3) moving only right or up.", "answer": 20},
    {"id": "201", "problem": "Find the sum of the first 20 even numbers.", "answer": 420},
    {"id": "202", "problem": "Find the volume of a cube with side length 5.", "answer": 125},
    {"id": "203", "problem": "Find the number of solutions to \\sin x = 0 for $x \\in [0, 2\\pi]$.", "answer": 3},
    {"id": "204", "problem": "Find the remainder when $10^{10}$ is divided by 7.", "answer": 4},
    {"id": "205", "problem": "Find the sum of the first 446 positive integers.", "answer": 99681},
    {"id": "206", "problem": "Find the integer $x$ such that \\log_2(x) + \\log_2(x-3) = 2.", "answer": 4},
    {"id": "207", "problem": "Find the sum of the first 10 terms of the geometric series 3, 6, 12, 24, \\dots", "answer": 3069},
    {"id": "208", "problem": "In $\\triangle ABC$, $AB=3, BC=4, AC=5$. Find the length of the altitude to $AC$. Multiply by 10.", "answer": 24},
    {"id": "209", "problem": "Find the number of subsets of \\{1, 2, 3, 4, 5, 6, 7\\} that contain exactly 3 elements.", "answer": 35},
    {"id": "210", "problem": "Evaluate \\sin^2(10^\\circ) + \\sin^2(20^\\circ) + \\dots + \\sin^2(80^\\circ) + \\sin^2(90^\\circ). Multiply by 10.", "answer": 50},
    {"id": "211", "problem": "Find the coefficient of $x^7$ in $(1+x)^{20}$.", "answer": 77520},
    {"id": "212", "problem": "How many ways to choose 3 numbers from {1, 2, ..., 20} such that no two are consecutive?", "answer": 816},
    {"id": "213", "problem": "Find the number of integers $n \\le 100$ that are not divisible by 2, 3, or 5.", "answer": 26},
    {"id": "214", "problem": "A committee of 5 is chosen from 6 men and 4 women. Number of ways if at least 3 are women?", "answer": 66},
    {"id": "215", "problem": "Find the number of non-negative integer solutions to $x_1+x_2+x_3+x_4=20$.", "answer": 1771},
    {"id": "216", "problem": "Alice flips a coin until 2 Heads in a row. Expected number of flips?", "answer": 6},
    {"id": "217", "problem": "What is the largest integer $n$ such that $3^n$ divides $100!$?", "answer": 48},
    {"id": "218", "problem": "A bag has 2 Red and 1 Blue. Draw with replacement until Blue. Expected draws?", "answer": 3},
    {"id": "219", "problem": "Probability a random point in a unit square is within 0.5 of a corner. Multiply by $(400/\\pi)$.", "answer": 100},
    {"id": "220", "problem": "Expected distance between two random points in [0, 10]. Multiply by 3.", "answer": 10},
    {"id": "221", "problem": "Find $a_{10}$ if $a_n = a_{n-1} + a_{n-2}$ with $a_1=2, a_2=2$.", "answer": 110},
    {"id": "222", "problem": "Sum of $1/n^2$ from 1 to 10. Multiply by 100 and floor.", "answer": 154},
    {"id": "223", "problem": "Find the number of diagonals in a convex 20-gon.", "answer": 170},
    {"id": "224", "problem": "Let $a_n$ be a sequence where $a_1=1, a_{n+1}=3a_n+2$. Find $a_6$.", "answer": 485},
    {"id": "225", "problem": "Find the coefficient of $x^3$ in the expansion of $(2x + 1)^6$.", "answer": 160},
    {"id": "226", "problem": "Find the number of positive integers $n \\le 100$ such that \\gcd(n, 100)=1.", "answer": 40},
    {"id": "227", "problem": "Find the smallest $n > 0$ such that $2^n \\equiv 1 \\pmod{17}$.", "answer": 8},
    {"id": "228", "problem": "Find the number of divisors of $10!$.", "answer": 270},
    {"id": "229", "problem": "Find the smallest prime factor of $2^{10}+1$.", "answer": 5},
    {"id": "230", "problem": "How many integers $n \\in [1, 1000]$ are multiples of 7 but not 49?", "answer": 122},
    {"id": "231", "problem": "Find the product of the roots of $x^4 + x^3 + x^2 + x + 1 = 0$.", "answer": 1},
    {"id": "232", "problem": "If $a+b=5$ and $a^2+b^2=13$, find $a^3+b^3$.", "answer": 35},
    {"id": "233", "problem": "Find the number of real roots of $x^6 - 3x^4 + 3x^2 - 1 = 0$.", "answer": 2},
    {"id": "234", "problem": "If $x$ is a complex number such that $x^2 + x + 1 = 0$, find $x^{99}$.", "answer": 1},
    {"id": "235", "problem": "Evaluate the limit as $x \\to 2$ of \\frac{x^3 - 8}{x - 2}.", "answer": 12},
    {"id": "236", "problem": "Find the sum of the prime numbers between 40 and 60.", "answer": 243},
    {"id": "237", "problem": "If $x, y, z > 0$ and $xyz=1$, find min of $x+2y+4z$. Multiply by 10 and floor.", "answer": 60},
    {"id": "238", "problem": "Find the maximum of \\sin x \\cos x. Multiply by 100.", "answer": 50},
    {"id": "239", "problem": "Find the min of $x + 4/x$ for $x > 0$.", "answer": 4},
    {"id": "240", "problem": "Find the value of $100 \\times \\max(xy)$ if $x^2+y^2=2$.", "answer": 100},
    {"id": "241", "problem": "A nim game has two piles: 3 and 4. Who wins? (1-A, 2-B).", "answer": 1},
    {"id": "242", "problem": "Find the nim-sum of 3, 5, and 6.", "answer": 0},
    {"id": "243", "problem": "In a game, Alice chooses X in [1, 100]. Bob guesses. If Bob guesses Y, Alice says high/low. Min guesses to win?", "answer": 7},
    {"id": "244", "problem": "Number of states in a 3x3 Tic-Tac-Toe where only 2 X's and 2 O's are placed.", "answer": 756},
    {"id": "245", "problem": "Find f(2) if f(x+1) = f(x)^2 for f(0)=2.", "answer": 16},
    {"id": "246", "problem": "If f(f(f(x))) = x and f(1)=2, f(2)=3, find f(3).", "answer": 1},
    {"id": "247", "problem": "If f(x) = f(x-1) + f(x-2) and f(0)=0, f(1)=1, find f(7).", "answer": 13},
    {"id": "248", "problem": "Find $f(5)$ if $f(x+y) = f(x)f(y)$ and $f(1)=2$.", "answer": 32},
    {"id": "249", "problem": "Find the number of edges in a $K_5$ complete graph.", "answer": 10},
    {"id": "250", "problem": "Number of vertices in a soccer ball (truncated icosahedron).", "answer": 60}
]

VERY_DIFFICULT_POOL = [
    {"id": "251", "problem": "Find the remainder when the number of non-negative integer solutions to the equation $a + b + c + d = 5000$ is divided by 100000.", "answer": 42501},
    {"id": "252", "problem": "A sequence is defined by $a_0 = 0$, $a_1 = 1$, and $a_n = 5a_{n-1} - 6a_{n-2}$ for $n \\ge 2$. Find the remainder when $a_{10^9}$ is divided by 10000.", "answer": 625},
    {"id": "253", "problem": "Find the number of positive integer divisors of $10^{99}$ that are perfect squares.", "answer": 2500},
    {"id": "254", "problem": "Find the number of integers $n \\le 1000$ such that $n$ is divisible by \\phi(n).", "answer": 34},
    {"id": "255", "problem": "Find the remainder when the sum $1! + 2! + 3! + \\dots + 10^6!$ is divided by 1000.", "answer": 313},
    {"id": "256", "problem": "A fair coin is flipped until three consecutive heads appear. Find the expected number of flips.", "answer": 14},
    {"id": "257", "problem": "Find the smallest positive integer $n$ such that $n^2 + (n+1)^2$ is a perfect square and $n > 100$.", "answer": 119},
    {"id": "258", "problem": "Find the number of ordered pairs of integers $(x,y)$ such that $x^2 + y^2 = 625$.", "answer": 20},
    {"id": "259", "problem": "Find the sum of all digits in the integer result of $2^{1000}$.", "answer": 1366},
    {"id": "260", "problem": "Find the remainder when \\binom{2000}{1000} is divided by 7.", "answer": 0},
    {"id": "261", "problem": "The sum of the first $n$ terms of a sequence is $S_n = n(n+1)(n+2)$. Find the value of $a_{100}$.", "answer": 30300},
    {"id": "262", "problem": "Find the last two digits of $7^{7^7}$.", "answer": 43},
    {"id": "263", "problem": "Find the smallest $n > 0$ such that $n^2$ ends in the digits 444.", "answer": 38},
    {"id": "264", "problem": "How many ways can 3 red, 3 blue, and 3 green balls be arranged in a row such that no two adjacent balls are the same color?", "answer": 174},
    {"id": "265", "problem": "Find the remainder when $1! + 2! + \\dots + 100!$ is divided by 15.", "answer": 3},
    {"id": "266", "problem": "Find the number of ordered triples $(x,y,z)$ of positive integers such that $x+y+z=20$ and they form the side lengths of a non-degenerate triangle.", "answer": 36},
    {"id": "267", "problem": "Find the sum of the squares of the first 20 positive integers.", "answer": 2870},
    {"id": "268", "problem": "Find the number of trailing zeros in $2024!$.", "answer": 503},
    {"id": "269", "problem": "Find the sum of all $n$ such that $2^n + 1$ is a Fermat prime and $n \\le 100$.", "answer": 31},
    {"id": "270", "problem": "Find the smallest positive integer $n$ such that \\sum_{i=1}^n i > 1,000,000.", "answer": 1414},
    {"id": "271", "problem": "If $x+y+z=1$ and $1/x + 1/y + 1/z = 0$, find $x^2 + y^2 + z^2$.", "answer": 1},
    {"id": "272", "problem": "Find the remainder when $2^{2^{10}}$ is divided by 17.", "answer": 1},
    {"id": "273", "problem": "Find the number of divisors of $120!$ that are prime.", "answer": 30},
    {"id": "274", "problem": "Find the value of $111,111,111^2$ and sum its digits.", "answer": 81},
    {"id": "275", "problem": "How many ways can a 2x10 grid be tiled with 1x2 dominoes?", "answer": 89},
    {"id": "276", "problem": "Find the smallest $n$ such that $n!$ has at least 290 trailing zeros.", "answer": 1170},
    {"id": "277", "problem": "Find the remainder when the number of subsets of \\{1, 2, \\dots, 20\\} that contain at least one prime number is divided by 100000.", "answer": 44480},
    {"id": "278", "problem": "Find the number of 4x4 binary matrices with exactly one 1 in each row and column.", "answer": 24},
    {"id": "279", "problem": "Find the smallest $n > 1$ such that $n$ is both a perfect square and a perfect cube.", "answer": 64},
    {"id": "280", "problem": "Find the sum of the digits of the multiplicative inverse of $7 \\pmod{100}$.", "answer": 7},
    {"id": "281", "problem": "Find the sum of all positive integer divisors of 1000.", "answer": 2340},
    {"id": "282", "problem": "Find the remainder when $2023^{2024}$ is divided by 1000.", "answer": 41},
    {"id": "283", "problem": "Find the value of $L_{10}$ in the Lucas sequence $L_0=2, L_1=1, L_n = L_{n-1} + L_{n-2}$.", "answer": 123},
    {"id": "284", "problem": "Find the number of paths from (0,0) to (4,4) moving only Right or Up that never cross the line $y=x$.", "answer": 14},
    {"id": "285", "problem": "Find the number of three-digit numbers $abc$ such that $a+b+c = 10$.", "answer": 54},
    {"id": "286", "problem": "Find the largest $k$ such that $3^k$ divides $100!$.", "answer": 48},
    {"id": "287", "problem": "Find the sum of the first 10 terms of the sequence $1, 2, 2, 3, 3, 3, 4, 4, 4, 4$.", "answer": 30},
    {"id": "288", "problem": "Find the number of integers $n \\le 1000$ such that $n^2 + 1$ is divisible by 5.", "answer": 400},
    {"id": "289", "problem": "Find the units digit of $3^{4024} + 4^{3024} \\pmod{10}$.", "answer": 7},
    {"id": "290", "problem": "Find the number of ways to choose 3 squares on a 4x4 chessboard such that no two are in the same row or column.", "answer": 96},
    {"id": "291", "problem": "Find the largest prime factor of $3^{10} + 1$.", "answer": 1181},
    {"id": "292", "problem": "Find the sum of the digits of $(10^{15} + 1)^2$.", "answer": 4},
    {"id": "293", "problem": "Find the number of ways to arrange the letters in 'AIMO' such that no letter is in its original position.", "answer": 9},
    {"id": "294", "problem": "Find the area of a regular hexagon with side length 4.", "answer": 42},
    {"id": "295", "problem": "Find the number of non-congruent triangles with integer sides and perimeter 15.", "answer": 7},
    {"id": "296", "problem": "Find the smallest $n$ such that $n/2$ is a square and $n/3$ is a cube.", "answer": 648},
    {"id": "297", "problem": "Find the remainder when $11^{121}$ is divided by 100.", "answer": 11},
    {"id": "298", "problem": "Find the number of divisors of $10^{10}$.", "answer": 121},
    {"id": "299", "problem": "Find the units digit of $7^{7^7} \\pmod{10}$.", "answer": 3},
    {"id": "300", "problem": "Find the value of $x$ such that \\sqrt{x + \\sqrt{x + \\sqrt{x + \\dots}}} = 1 + \\sqrt{x}.", "answer": 0}
]

# --- 🎯 Selection Logic ---

def select_problems():
    """
    Returns a list of problems based on user input or defaults.
    """
    global SET_SIZE
    test_problems = []
    
    # Detect if we are in an interactive environment
    is_interactive = sys.stdin.isatty() or os.environ.get("AIMO_INTERACTIVE") == "1"
    is_direct_run = (__name__ == "__main__")

    if not is_direct_run and not is_interactive:
        # Default values for when imported by lvties9.py in non-interactive mode (e.g. Kaggle)
        SET_SIZE = 50 
        n_easy, n_med, n_hard, n_vhard = 10, 10, 15, 15
        test_problems = (
            random.sample(EASY_POOL, min(n_easy, len(EASY_POOL))) +
            random.sample(MEDIUM_POOL, min(n_med, len(MEDIUM_POOL))) +
            random.sample(HARD_POOL, min(n_hard, len(HARD_POOL))) +
            random.sample(VERY_DIFFICULT_POOL, min(n_vhard, len(VERY_DIFFICULT_POOL)))
        )
        random.shuffle(test_problems)
        return test_problems

    # Interactive Selection
    try:
        print("\n" + "="*50)
        print("📊 --- STRATIFIED BENCHMARK SELECTOR ---")
        print("="*50)
        print(" [0]  Exit Selection")
        print(" [N]  Standard Mixed: 3, 5, 10, 20, 30, 40, 50")
        print("\n [Pool Shortcuts]:")
        print("  - 1s: Easy Pool (1, 11, 111, 1111...)")
        print("  - 2s: Medium Pool (2, 22, 222, 2222...)")
        print("  - 8s: Hard Pool (8, 88, 888, 8888...)")
        print("  - 9s: Very Difficult Pool (9, 99, 999, 9999...)")
        print("\n [Specific ID]:")
        print("  - i[number] (e.g. i1, i10, i250)")
        print("="*50)
        
        user_input = input("Selection: ").strip()
        if not user_input or user_input == '0':
            return []

        vhard_shortcuts = {"9": 1, "99": 2, "999": 3, "9999": 4, "99999": 5}
        easy_shortcuts = {"1": 1, "11": 2, "111": 3, "1111": 4, "11111": 5}
        medium_shortcuts = {"2": 1, "22": 2, "222": 3, "2222": 4, "22222": 5}
        hard_shortcuts = {"8": 1, "88": 2, "888": 3, "8888": 4, "88888": 5}
        
        if user_input.lower().startswith('i') and len(user_input) > 1:
            id_str = user_input.lower().replace('i', '').strip()
            id_formatted = id_str.zfill(2) if id_str.isdigit() else id_str
            all_pools = EASY_POOL + MEDIUM_POOL + HARD_POOL + VERY_DIFFICULT_POOL
            matched_problem = [p for p in all_pools if p['id'] == id_formatted or p['id'].lower() == id_str.lower()]
            if matched_problem:
                return matched_problem[:1]
        
        elif user_input in vhard_shortcuts:
            return random.sample(VERY_DIFFICULT_POOL, min(vhard_shortcuts[user_input], len(VERY_DIFFICULT_POOL)))
        elif user_input in easy_shortcuts:
            return random.sample(EASY_POOL, min(easy_shortcuts[user_input], len(EASY_POOL)))
        elif user_input in medium_shortcuts:
            return random.sample(MEDIUM_POOL, min(medium_shortcuts[user_input], len(MEDIUM_POOL)))
        elif user_input in hard_shortcuts:
            return random.sample(HARD_POOL, min(hard_shortcuts[user_input], len(HARD_POOL)))
        elif user_input.isdigit():
            size = int(user_input)
            if size == 3: n_easy, n_med, n_hard, n_vhard = 1, 1, 0, 1
            elif size == 5: n_easy, n_med, n_hard, n_vhard = 1, 1, 1, 2
            elif size == 10: n_easy, n_med, n_hard, n_vhard = 2, 3, 2, 3
            else:
                n_easy = max(1, int(size * 0.2))
                n_med = max(1, int(size * 0.2))
                n_hard = max(1, int(size * 0.3))
                n_vhard = size - n_easy - n_med - n_hard
            
            res = (
                random.sample(EASY_POOL, min(n_easy, len(EASY_POOL))) +
                random.sample(MEDIUM_POOL, min(n_med, len(MEDIUM_POOL))) +
                random.sample(HARD_POOL, min(n_hard, len(HARD_POOL))) +
                random.sample(VERY_DIFFICULT_POOL, min(n_vhard, len(VERY_DIFFICULT_POOL)))
            )
            random.shuffle(res)
            return res

    except Exception as e:
        print(f"⚠️ Error during selection: {e}")
    
    return random.sample(EASY_POOL, 5)

# Initialize for import compatibility
TEST_PROBLEMS = select_problems()

# --- 🚀 Standalone Preview & Loop ---
if __name__ == "__main__":
    while True:
        if TEST_PROBLEMS:
            print(f"\n✅ Working with {len(TEST_PROBLEMS)} problems.")
            for i, prob in enumerate(TEST_PROBLEMS):
                print(f" [{i+1}/{len(TEST_PROBLEMS)}] ID: {prob['id']} | Problem: {prob['problem'][:80]}...")
        else:
            print("\n🚪 Selection cleared.")

        print("\n" + "-"*40)
        choice = input("🔄 Choose another set? (Enter to re-select, or 'q' to quit): ").strip().lower()
        if choice == 'q':
            break
        
        TEST_PROBLEMS = select_problems()


Writing test.py


In [3]:
%%writefile test.py
import os
import random
import sys

# --- CONFIGURATION ---
SET_SIZE = 50 # Default for non-interactive benchmark

# --- 🧠 Problem Bank (Tiers) ---
EASY_POOL = [
    {"id": "01", "problem": "Find the sum of all prime numbers $p$ such that $p+2$ and $p+4$ are also prime.", "answer": 3},
    {"id": "02", "problem": "Let $P(x) = x^2 - 10x + 25$. Find the value of $P(15)$.", "answer": 100},
    {"id": "03", "problem": "Find the number of positive integers $n < 100$ such that $n$ is a multiple of 3 and $n+1$ is a multiple of 4.", "answer": 9},
    {"id": "04", "problem": "If $a$ and $b$ are roots of $x^2 - 14x + 48 = 0$, find $a^2 + b^2$.", "answer": 100},
    {"id": "05", "problem": "Find the sum of the digits of the integer $10^{10} - 1$.", "answer": 90},
    {"id": "06", "problem": "A sequence $a_n$ is defined by $a_1 = 3$ and $a_{n+1} = a_n^2 - 2$. Find $a_3$.", "answer": 47},
    {"id": "07", "problem": "Find the number of ordered pairs of integers $(x,y)$ such that $x^2 + y^2 = 25$.", "answer": 12},
    {"id": "08", "problem": "Find the sum of all real roots of $(x-1)(x-2)(x-3) = 0$.", "answer": 6},
    {"id": "09", "problem": "Evaluate the infinite sum $\\sum_{n=1}^\\infty \\frac{n}{3^n}$. Express your answer as $m/n$, and find $10m+n$.", "answer": 34},
    {"id": "10", "problem": "Find the smallest positive integer $n$ such that $n!$ is divisible by 1000.", "answer": 15},
    {"id": "11", "problem": "Find the number of integers $x \\in \\{1, 2, \\dots, 1000\\}$ that are relatively prime to 10.", "answer": 400},
    {"id": "12", "problem": "Find the constant term in the expansion of $(x + \\frac{2}{x})^6$.", "answer": 160},
    {"id": "13", "problem": "Find the number of three-digit integers $abc$ such that $a < b < c$.", "answer": 84},
    {"id": "14", "problem": "Find the smallest positive integer $n$ such that $2^n \\equiv 1 \\pmod{31}$.", "answer": 5},
    {"id": "15", "problem": "Evaluate \\sqrt{12 + \\sqrt{12 + \\sqrt{12 + \\dots}}}.", "answer": 4},
    {"id": "16", "problem": "Find the sum of the first 100 terms of the arithmetic progression $5, 8, 11, \\dots$.", "answer": 15350},
    {"id": "17", "problem": "Find the number of subsets of \\{1, 2, 3, 4, 5, 6, 7, 8\\} that do not contain two consecutive integers.", "answer": 55},
    {"id": "18", "problem": "Find the sum of the squares of the coefficients of $(x+1)^4$.", "answer": 70},
    {"id": "19", "problem": "Find the number of positive integers $n$ for which $n^2 + 8n + 15$ is prime.", "answer": 0},
    {"id": "20", "problem": "Find the remainder when $3^{2024}$ is divided by 100.", "answer": 81},
    {"id": "21", "problem": "Evaluate $100^2 - 99^2 + 98^2 - 97^2 + \\dots + 2^2 - 1^2$.", "answer": 5050},
    {"id": "22", "problem": "Find the value of $n$ such that \\binom{n}{2} = 4950.", "answer": 100},
    {"id": "23", "problem": "Find the number of ways to arrange the letters in the word 'KAGGLE'.", "answer": 360},
    {"id": "24", "problem": "A circle is inscribed in a right triangle with sides 3, 4, and 5. Find the square of the area of the circle multiplied by $(100/\\pi)$.", "answer": 314},
    {"id": "25", "problem": "Find the sum of the zeros of $P(x) = x^3 - 6x^2 + 11x - 6$.", "answer": 6},
    {"id": "26", "problem": "Find the smallest integer $n > 1$ such that $n^2$ is a cube and $n^3$ is a square.", "answer": 64},
    {"id": "27", "problem": "Find the number of divisors of $2^5 \\cdot 3^4 \\cdot 5^3$.", "answer": 120},
    {"id": "28", "problem": "Find the sum of the interior angles (in degrees) of a convex decagon.", "answer": 1440},
    {"id": "29", "problem": "How many positive integers $n < 500$ are not divisible by 2, 3, or 5?", "answer": 134},
    {"id": "30", "problem": "Find the value of $x$ satisfying \\log_2(\\log_3(\\log_4 x)) = 0.", "answer": 64},
    {"id": "31", "problem": "A square has a perimeter of 40. Find its area.", "answer": 100},
    {"id": "32", "problem": "Find the hypotenuse of a right triangle with legs 8 and 15.", "answer": 17},
    {"id": "33", "problem": "Find the sum of the interior angles of a hexagon in degrees.", "answer": 720},
    {"id": "34", "problem": "A circle has area $49\\pi$. Find its diameter.", "answer": 14},
    {"id": "35", "problem": "Find the area of a triangle with base 10 and height 12.", "answer": 60},
    {"id": "36", "problem": "How many ways can 3 people be seated in 3 chairs?", "answer": 6},
    {"id": "37", "problem": "Find the number of subsets of {1, 2, 3, 4}.", "answer": 16},
    {"id": "38", "problem": "How many ways to choose 2 colors from 5 distinct colors?", "answer": 10},
    {"id": "39", "problem": "A bag has 3 red and 2 blue marbles. How many ways to pick 2 marbles (order matters)?", "answer": 20},
    {"id": "40", "problem": "Find the number of permutations of the letters in 'MATH'.", "answer": 24},
    {"id": "41", "problem": "A fair coin is flipped twice. Find the probability of 2 heads. Multiply by 100.", "answer": 25},
    {"id": "42", "problem": "A die is rolled. Find the probability of rolling a 1 or 2. Multiply by 60.", "answer": 20},
    {"id": "43", "problem": "If the probability of rain is 0.3, find the prob. it doesn't rain. Multiply by 100.", "answer": 70},
    {"id": "44", "problem": "A bag has 10 balls. 3 are red. Prob of picking a red? Multiply by 100.", "answer": 30},
    {"id": "45", "problem": "Expected value of a die roll is 3.5. Multiply by 10.", "answer": 35},
    {"id": "46", "problem": "Find the 5th term of the sequence 2, 5, 8, 11...", "answer": 14},
    {"id": "47", "problem": "Find the sum of 1, 2, 4, 8, 16.", "answer": 31},
    {"id": "48", "problem": "Find the common ratio of the geometric sequence 3, 6, 12, 24.", "answer": 2},
    {"id": "49", "problem": "Find the 10th term of $a_n = 2n + 1$.", "answer": 21},
    {"id": "50", "problem": "Sum of first 10 positive integers.", "answer": 55},
    {"id": "51", "problem": "Find the greatest common divisor of 12 and 18.", "answer": 6},
    {"id": "52", "problem": "Find the remainder when 100 is divided by 7.", "answer": 2},
    {"id": "53", "problem": "Find the smallest prime number greater than 10.", "answer": 11},
    {"id": "54", "problem": "Find the number of divisors of 12.", "answer": 6},
    {"id": "55", "problem": "What is the 5th prime number?", "answer": 11},
    {"id": "56", "problem": "Solve for x: 3x + 5 = 20.", "answer": 5},
    {"id": "57", "problem": "If f(x) = x^2 + 1, find f(3).", "answer": 10},
    {"id": "58", "problem": "Factor x^2 - 9. Find the positive root.", "answer": 3},
    {"id": "59", "problem": "Find the slope of the line y = 4x + 7.", "answer": 4},
    {"id": "60", "problem": "Evaluate (2+3)^2 - 5.", "answer": 20},
    {"id": "61", "problem": "Find the smallest integer x such that x > 5.5.", "answer": 6},
    {"id": "62", "problem": "Find the maximum value of 10 - x^2.", "answer": 10},
    {"id": "63", "problem": "If x < 5 and x is an integer, find the largest x.", "answer": 4},
    {"id": "64", "problem": "Solve for x: |x| = 7 and x < 0. Express |x|.", "answer": 7},
    {"id": "65", "problem": "Is 3 > 2? (1 for Yes, 0 for No).", "answer": 1},
    {"id": "66", "problem": "If Alice starts with 10 apples and gives 3 to Bob, how many left?", "answer": 7},
    {"id": "67", "problem": "In a game, the winner gets 5 points. If Bob wins 3 times, how many points?", "answer": 15},
    {"id": "68", "problem": "A game has 2 players. If Alice wins, Bob loses. Result is 1. If tie 0. If Alice wins, result?", "answer": 1},
    {"id": "69", "problem": "Two people flip a coin. Alice wins on Heads. Heads occurs. Who wins? (1 for Alice, 2 for Bob).", "answer": 1},
    {"id": "70", "problem": "If a strategy guarantees a win in 5 moves, how many moves to win?", "answer": 5},
    {"id": "71", "problem": "If f(x) = 2x, find f(10).", "answer": 20},
    {"id": "72", "problem": "If f(x) = f(x+1) - 1 and f(0) = 0, find f(5).", "answer": 5},
    {"id": "73", "problem": "If f(f(x)) = x and f(1) = 2, find f(2).", "answer": 1},
    {"id": "74", "problem": "A graph has 4 vertices and 0 edges. Find the number of vertices.", "answer": 4},
    {"id": "75", "problem": "Find the number of edges in a triangle graph.", "answer": 3}
]

MEDIUM_POOL = [
    {"id": "76", "problem": "Find the coefficient of $x^7$ in $(1+x)^{10}$.", "answer": 120},
    {"id": "77", "problem": "Find the number of solutions to $x_1 + x_2 + x_3 = 10$ in non-negative integers.", "answer": 66},
    {"id": "78", "problem": "Evaluate \\sum_{k=1}^{10} (k^3 - k).", "answer": 2970},
    {"id": "79", "problem": "Find the number of trailing zeros in $100!$.", "answer": 24},
    {"id": "80", "problem": "Find the sum of all three-digit palindromes.", "answer": 49500},
    {"id": "81", "problem": "If \\tan x = 3/4, find the value of $100 \\sin(2x)$.", "answer": 96},
    {"id": "82", "problem": "Find the smallest positive integer $n$ such that $2^n > n^{10}$.", "answer": 59},
    {"id": "83", "problem": "Find the number of positive integers $n \\le 1000$ such that \\lfloor \\sqrt{n} \\rfloor divides $n$.", "answer": 92},
    {"id": "84", "problem": "Find the value of $1 \\cdot 1! + 2 \\cdot 2! + \\dots + 6 \\cdot 6!$.", "answer": 5039},
    {"id": "85", "problem": "Find the product of the roots of $x^4 - 5x^2 + 4 = 0$.", "answer": 4},
    {"id": "86", "problem": "How many ways can 6 people be seated around a circular table (reflections distinct)?", "answer": 120},
    {"id": "87", "problem": "Find the sum of the first 50 odd numbers.", "answer": 2500},
    {"id": "88", "problem": "Find the value of $1234^2 - 1233 \\cdot 1235$.", "answer": 1},
    {"id": "89", "problem": "Find the number of integer points $(x,y)$ inside or on the circle $x^2 + y^2 = 10$.", "answer": 37},
    {"id": "90", "problem": "Find the last digit of $7^{7^7}$.", "answer": 3},
    {"id": "91", "problem": "A fair coin is flipped 10 times. Find the number of outcomes with exactly 5 heads.", "answer": 252},
    {"id": "92", "problem": "Find the product of all distinct prime factors of 2024.", "answer": 506},
    {"id": "93", "problem": "Find the value of $x$ if $x + \\frac{1}{x} = 3$, find $x^4 + \\frac{1}{x^4}$.", "answer": 47},
    {"id": "94", "problem": "Find the sum of all integers from 1 to 446.", "answer": 99681},
    {"id": "95", "problem": "Find the largest 5-digit palindrome divisible by 9.", "answer": 99999},
    {"id": "96", "problem": "Find the number of integers $n$ such that $n+3$ divides $n^2+7$.", "answer": 10},
    {"id": "97", "problem": "Find the sum of all real roots of $x^2 - 100x + 99 = 0$.", "answer": 100},
    {"id": "98", "problem": "Find the number of subsets of \\{1, 2, \\dots, 10\\} with exactly 3 elements such that no two elements are consecutive.", "answer": 56},
    {"id": "99", "problem": "Find the remainder when $2^{2025}$ is divided by 13.", "answer": 5},
    {"id": "100", "problem": "Find the value of \\sum_{k=1}^{100} \\lfloor \\log_2 k \\rfloor.", "answer": 480},
    {"id": "101", "problem": "How many positive integers divide at least two of the integers 60, 126, and 140?", "answer": 10},
    {"id": "102", "problem": "Find the number of ordered triples $(a,b,c)$ of positive integers such that $a+b+c = 15$.", "answer": 91},
    {"id": "103", "problem": "Find the sum of all digits of $11^{11}$.", "answer": 41},
    {"id": "104", "problem": "Find the absolute difference between the roots of $x^2 - 100x + 99 = 0$.", "answer": 98},
    {"id": "105", "problem": "Find the number of solutions to $x^2 \\equiv 1 \\pmod{100}$.", "answer": 4},
    {"id": "106", "problem": "Find the area of a triangle with side lengths 13, 14, and 15.", "answer": 84},
    {"id": "107", "problem": "Evaluate \\int_0^2 (x^3 - 3x^2 + 4) dx. Multiply by 10.", "answer": 40},
    {"id": "108", "problem": "Find the smallest $n$ such that $1+2+\\dots+n > 1000$.", "answer": 45},
    {"id": "109", "problem": "Find the number of ways to color the edges of a square with 3 colors s.t. adjacent edges have different colors.", "answer": 18},
    {"id": "110", "problem": "Find the largest prime factor of $3^{10} - 1$.", "answer": 61},
    {"id": "111", "problem": "Find the value of $x^3+y^3$ if $x+y=5$ and $xy=6$.", "answer": 35},
    {"id": "112", "problem": "Find the number of 4-digit numbers where all digits are different and odd.", "answer": 120},
    {"id": "113", "problem": "Find the minimum value of $x^2 - 12x + 40$.", "answer": 4},
    {"id": "114", "problem": "Find the sum of all $n$ such that $n$ is a divisor of $100$ but not a divisor of $50$.", "answer": 124},
    {"id": "115", "problem": "Find the number of ways to stack 4 identical blocks in 3 different bins.", "answer": 15},
    {"id": "116", "problem": "A right triangle has legs of length 10 and 24. Find the radius of its circumcircle.", "answer": 13},
    {"id": "117", "problem": "Find the length of the diagonal of a rectangular box with dimensions 1, 2, and 2.", "answer": 3},
    {"id": "118", "problem": "A circle is tangent to both axes in the first quadrant and passes through (2, 4). Find the larger possible radius.", "answer": 10},
    {"id": "119", "problem": "Find the sum of the first 15 terms of the arithmetic sequence 2, 9, 16, 23, ...", "answer": 765},
    {"id": "120", "problem": "If $\\sin x + \\cos x = 1.2$, find $100 \\sin(2x)$.", "answer": 44},
    {"id": "121", "problem": "Find the volume of a sphere with radius 3. Multiply by $(3/\\pi)$.", "answer": 108},
    {"id": "122", "problem": "Find the number of ways to choose 4 items from 10.", "answer": 210},
    {"id": "123", "problem": "Find the number of anagrams of 'BANANA'.", "answer": 60},
    {"id": "124", "problem": "How many ways to partition 6 items into two groups of 3?", "answer": 10},
    {"id": "125", "problem": "Find the number of integer solutions to x+y+z=10 with x,y,z > 0.", "answer": 36},
    {"id": "126", "problem": "A grid of 3x3 dots. How many squares can be formed?", "answer": 6},
    {"id": "127", "problem": "How many ways to seat 5 people in a line if Alice and Bob must be adjacent?", "answer": 48},
    {"id": "128", "problem": "A fair coin is flipped 3 times. What is the expected number of heads? Multiply by 10.", "answer": 15},
    {"id": "129", "problem": "A die is rolled until a 6 appears. Expected number of rolls?", "answer": 6},
    {"id": "130", "problem": "A bag has 5 red and 5 blue balls. If you draw 2, expected red? Multiply by 10.", "answer": 10},
    {"id": "131", "problem": "Two dice are rolled. Sum is X. Expected value of X?", "answer": 7},
    {"id": "132", "problem": "A coin is flipped. Heads you get 10 points, Tails 0. Expected points?", "answer": 5},
    {"id": "133", "problem": "Probability of rain is 1/2. Expected days of rain in a week? Multiply by 2.", "answer": 7},
    {"id": "134", "problem": "Find the sum of the first 20 terms of $a_n = n^2$.", "answer": 2870},
    {"id": "135", "problem": "Find the limit of $n/(2n+1)$ as n goes to infinity. Multiply by 10.", "answer": 5},
    {"id": "136", "problem": "Find the 10th Fibonacci number starting from $F_1=1, F_2=1$.", "answer": 55},
    {"id": "137", "problem": "Find the sum of the geometric series $1 + 1/2 + 1/4 + \\dots$. Multiply by 10.", "answer": 20},
    {"id": "138", "problem": "Sequence $a_1=1, a_n = 2a_{n-1} + 1$. Find $a_6$.", "answer": 63},
    {"id": "139", "problem": "Find the product of the first 5 terms of $a_n = 2^n$.", "answer": 32768},
    {"id": "140", "problem": "Find the smallest $n$ such that $n \\equiv 1 \\pmod 3$ and $n \\equiv 2 \\pmod 5$.", "answer": 7},
    {"id": "141", "problem": "Find the remainder of $123456 \\pmod{11}$.", "answer": 3},
    {"id": "142", "problem": "Find the number of digits in $2^{10} \\cdot 5^7$.", "answer": 8},
    {"id": "143", "problem": "Find the largest prime factor of 1001.", "answer": 13},
    {"id": "144", "problem": "Find the number of positive integers $n < 100$ divisible by both 4 and 6.", "answer": 8},
    {"id": "145", "problem": "If $x$ is an integer and $x^2 \\equiv 4 \\pmod 7$, find the smallest positive $x$.", "answer": 2},
    {"id": "146", "problem": "Find the sum of the roots of $2x^2 - 100x + 1 = 0$.", "answer": 50},
    {"id": "147", "problem": "If $P(x) = x^3 - 12x^2 + 47x - 60$, find the product of its roots.", "answer": 60},
    {"id": "148", "problem": "Solve for x: $x + 2\\sqrt{x} = 15$.", "answer": 9},
    {"id": "149", "problem": "Find the value of $k$ if $x=2$ is a root of $x^2 + kx - 10 = 0$.", "answer": 3},
    {"id": "150", "problem": "Evaluate $(1+i)^8$.", "answer": 16},
    {"id": "151", "problem": "Find the vertex of $y = x^2 - 4x + 7$. Express the y-coordinate.", "answer": 3},
    {"id": "152", "problem": "Find the minimum value of $x + 1/x$ for $x > 0$.", "answer": 2},
    {"id": "153", "problem": "If $x+y=10$, find the maximum of $xy$.", "answer": 25},
    {"id": "154", "problem": "Find the smallest $x$ such that $2x^2 - 8x + 10 \\le 2$.", "answer": 2},
    {"id": "155", "problem": "If $a, b > 0$ and $a+b=6$, find max of $a^2b$.", "answer": 32},
    {"id": "156", "problem": "Find the sum of all integers x such that $x^2 - 5x + 6 \\le 0$.", "answer": 5},
    {"id": "157", "problem": "Find the minimum of $e^x + e^{-x}$. Round to nearest integer.", "answer": 2},
    {"id": "158", "problem": "Alice and Bob play a game. Alice starts with 10. Each turn they subtract 1. Last to 0 wins. Who wins? (1-A, 2-B).", "answer": 2},
    {"id": "159", "problem": "In a game of 21, the sum starts at 0. Players add 1, 2, or 3. Player who makes it 21 wins. First player adds X. Find X to win.", "answer": 1},
    {"id": "160", "problem": "If $x$ and $y$ are positive integers such that $x^2 - y^2 = 51$, find the maximum possible value of $x+y$.", "answer": 51},
    {"id": "161", "problem": "Tic-Tac-Toe has how many squares?", "answer": 9},
    {"id": "162", "problem": "Alice wins if she flips Heads. Prob is 0.6. Expected wins in 100 flips?", "answer": 60},
    {"id": "163", "problem": "Total number of states in a 2-node game where each node can be 0 or 1.", "answer": 4},
    {"id": "164", "problem": "If f(x+y) = f(x) + f(y) and f(1) = 3, find f(10).", "answer": 30},
    {"id": "165", "problem": "If f(x) = ax + b, f(1)=5, f(2)=7, find f(10).", "answer": 23},
    {"id": "166", "problem": "If f(x) = f(x-1) + 2x - 1 and f(0) = 0, find f(4).", "answer": 16},
    {"id": "167", "problem": "If f(f(x)) = x+2 and f(0)=1, find f(1).", "answer": 2},
    {"id": "168", "problem": "If f(x^2) = x^4 + 1, find f(5).", "answer": 26},
    {"id": "169", "problem": "Evaluate the sum of the coefficients of the polynomial $P(x) = (3x^2 - 2x + 1)^4$.", "answer": 16},
    {"id": "170", "problem": "A graph has 10 vertices. Total possible edges?", "answer": 45},
    {"id": "171", "problem": "Find the degree of a vertex in a cycle graph $C_5$.", "answer": 2},
    {"id": "172", "problem": "Number of edges in a tree with 100 vertices.", "answer": 99},
    {"id": "173", "problem": "Maximum number of edges in a bipartite graph with 2+2 vertices.", "answer": 4},
    {"id": "174", "problem": "How many colors needed for a bipartite graph?", "answer": 2},
    {"id": "175", "problem": "Number of vertices in a cube.", "answer": 8}
]

HARD_POOL = [
    {"id": "176", "problem": "Find the value of \\sin(15^\\circ) \\cos(15^\\circ) multiplied by 400.", "answer": 100},
    {"id": "177", "problem": "Find the largest value of $x$ for which \\sqrt{x} + \\sqrt{x-5} = 5.", "answer": 9},
    {"id": "178", "problem": "Find the number of permutations of (1, 2, 3, 4, 5, 6) with no fixed points.", "answer": 265},
    {"id": "179", "problem": "Evaluate $2^{2^3} - (2^2)^3$.", "answer": 192},
    {"id": "180", "problem": "Find the radius of the incircle of a 3-4-5 triangle.", "answer": 1},
    {"id": "181", "problem": "Find the number of real solutions to $x^4 - 4x^2 + 4 = 0$.", "answer": 2},
    {"id": "182", "problem": "Find the greatest common divisor of $2^{10}-1$ and $2^{15}-1$.", "answer": 31},
    {"id": "183", "problem": "Find the sum of all integers $x$ such that |x-5| < 3.", "answer": 25},
    {"id": "184", "problem": "Find the value of $n$ if $1+3+5+\\dots+(2n-1) = 10000$.", "answer": 100},
    {"id": "185", "problem": "Find the units digit of $2^{2024}$.", "answer": 6},
    {"id": "186", "problem": "How many ways can 10 distinct books be given to 2 children such that each child gets 5?", "answer": 252},
    {"id": "187", "problem": "Find the number of three-digit numbers divisible by 7.", "answer": 128},
    {"id": "188", "problem": "Find the sum of the roots of $x^2 - 2024x + 2023 = 0$.", "answer": 2024},
    {"id": "189", "problem": "Find the remainder when $x^{100}$ is divided by $x-1$.", "answer": 1},
    {"id": "190", "problem": "Find the area of a circle with circumference $20\\pi$.", "answer": 314},
    {"id": "191", "problem": "Find the number of integers $n$ such that $n^2+1$ is prime and $n \\le 10$.", "answer": 5},
    {"id": "192", "problem": "Find the value of $1/2 + 1/4 + 1/8 + \\dots$ up to 10 terms, then multiply by 1024 and subtract from 1024.", "answer": 1},
    {"id": "193", "problem": "Find the sum of the digits of $123 \\times 456$.", "answer": 27},
    {"id": "194", "problem": "Find the number of ways to choose 2 numbers from \\{1, 2, \\dots, 20\\} whose sum is even.", "answer": 90},
    {"id": "195", "problem": "Find the smallest prime $p > 100$.", "answer": 101},
    {"id": "196", "problem": "Find the number of edges in a complete graph with 10 vertices.", "answer": 45},
    {"id": "197", "problem": "Evaluate $3^5 - 5^3$.", "answer": 118},
    {"id": "198", "problem": "Find the number of positive integers $k < 20$ such that $x^2 + kx + 16 = 0$ has real roots.", "answer": 12}, 
    {"id": "199", "problem": "Find the value of $111 \\times 111$.", "answer": 12321},
    {"id": "200", "problem": "Find the number of paths from (0,0) to (3,3) moving only right or up.", "answer": 20},
    {"id": "201", "problem": "Find the sum of the first 20 even numbers.", "answer": 420},
    {"id": "202", "problem": "Find the volume of a cube with side length 5.", "answer": 125},
    {"id": "203", "problem": "Find the number of solutions to \\sin x = 0 for $x \\in [0, 2\\pi]$.", "answer": 3},
    {"id": "204", "problem": "Find the remainder when $10^{10}$ is divided by 7.", "answer": 4},
    {"id": "205", "problem": "Find the sum of the first 446 positive integers.", "answer": 99681},
    {"id": "206", "problem": "Find the integer $x$ such that \\log_2(x) + \\log_2(x-3) = 2.", "answer": 4},
    {"id": "207", "problem": "Find the sum of the first 10 terms of the geometric series 3, 6, 12, 24, \\dots", "answer": 3069},
    {"id": "208", "problem": "In $\\triangle ABC$, $AB=3, BC=4, AC=5$. Find the length of the altitude to $AC$. Multiply by 10.", "answer": 24},
    {"id": "209", "problem": "Find the number of subsets of \\{1, 2, 3, 4, 5, 6, 7\\} that contain exactly 3 elements.", "answer": 35},
    {"id": "210", "problem": "Evaluate \\sin^2(10^\\circ) + \\sin^2(20^\\circ) + \\dots + \\sin^2(80^\\circ) + \\sin^2(90^\\circ). Multiply by 10.", "answer": 50},
    {"id": "211", "problem": "Find the coefficient of $x^7$ in $(1+x)^{20}$.", "answer": 77520},
    {"id": "212", "problem": "How many ways to choose 3 numbers from {1, 2, ..., 20} such that no two are consecutive?", "answer": 816},
    {"id": "213", "problem": "Find the number of integers $n \\le 100$ that are not divisible by 2, 3, or 5.", "answer": 26},
    {"id": "214", "problem": "A committee of 5 is chosen from 6 men and 4 women. Number of ways if at least 3 are women?", "answer": 66},
    {"id": "215", "problem": "Find the number of non-negative integer solutions to $x_1+x_2+x_3+x_4=20$.", "answer": 1771},
    {"id": "216", "problem": "Alice flips a coin until 2 Heads in a row. Expected number of flips?", "answer": 6},
    {"id": "217", "problem": "What is the largest integer $n$ such that $3^n$ divides $100!$?", "answer": 48},
    {"id": "218", "problem": "A bag has 2 Red and 1 Blue. Draw with replacement until Blue. Expected draws?", "answer": 3},
    {"id": "219", "problem": "Probability a random point in a unit square is within 0.5 of a corner. Multiply by $(400/\\pi)$.", "answer": 100},
    {"id": "220", "problem": "Expected distance between two random points in [0, 10]. Multiply by 3.", "answer": 10},
    {"id": "221", "problem": "Find $a_{10}$ if $a_n = a_{n-1} + a_{n-2}$ with $a_1=2, a_2=2$.", "answer": 110},
    {"id": "222", "problem": "Sum of $1/n^2$ from 1 to 10. Multiply by 100 and floor.", "answer": 154},
    {"id": "223", "problem": "Find the number of diagonals in a convex 20-gon.", "answer": 170},
    {"id": "224", "problem": "Let $a_n$ be a sequence where $a_1=1, a_{n+1}=3a_n+2$. Find $a_6$.", "answer": 485},
    {"id": "225", "problem": "Find the coefficient of $x^3$ in the expansion of $(2x + 1)^6$.", "answer": 160},
    {"id": "226", "problem": "Find the number of positive integers $n \\le 100$ such that \\gcd(n, 100)=1.", "answer": 40},
    {"id": "227", "problem": "Find the smallest $n > 0$ such that $2^n \\equiv 1 \\pmod{17}$.", "answer": 8},
    {"id": "228", "problem": "Find the number of divisors of $10!$.", "answer": 270},
    {"id": "229", "problem": "Find the smallest prime factor of $2^{10}+1$.", "answer": 5},
    {"id": "230", "problem": "How many integers $n \\in [1, 1000]$ are multiples of 7 but not 49?", "answer": 122},
    {"id": "231", "problem": "Find the product of the roots of $x^4 + x^3 + x^2 + x + 1 = 0$.", "answer": 1},
    {"id": "232", "problem": "If $a+b=5$ and $a^2+b^2=13$, find $a^3+b^3$.", "answer": 35},
    {"id": "233", "problem": "Find the number of real roots of $x^6 - 3x^4 + 3x^2 - 1 = 0$.", "answer": 2},
    {"id": "234", "problem": "If $x$ is a complex number such that $x^2 + x + 1 = 0$, find $x^{99}$.", "answer": 1},
    {"id": "235", "problem": "Evaluate the limit as $x \\to 2$ of \\frac{x^3 - 8}{x - 2}.", "answer": 12},
    {"id": "236", "problem": "Find the sum of the prime numbers between 40 and 60.", "answer": 243},
    {"id": "237", "problem": "If $x, y, z > 0$ and $xyz=1$, find min of $x+2y+4z$. Multiply by 10 and floor.", "answer": 60},
    {"id": "238", "problem": "Find the maximum of \\sin x \\cos x. Multiply by 100.", "answer": 50},
    {"id": "239", "problem": "Find the min of $x + 4/x$ for $x > 0$.", "answer": 4},
    {"id": "240", "problem": "Find the value of $100 \\times \\max(xy)$ if $x^2+y^2=2$.", "answer": 100},
    {"id": "241", "problem": "A nim game has two piles: 3 and 4. Who wins? (1-A, 2-B).", "answer": 1},
    {"id": "242", "problem": "Find the nim-sum of 3, 5, and 6.", "answer": 0},
    {"id": "243", "problem": "In a game, Alice chooses X in [1, 100]. Bob guesses. If Bob guesses Y, Alice says high/low. Min guesses to win?", "answer": 7},
    {"id": "244", "problem": "Number of states in a 3x3 Tic-Tac-Toe where only 2 X's and 2 O's are placed.", "answer": 756},
    {"id": "245", "problem": "Find f(2) if f(x+1) = f(x)^2 for f(0)=2.", "answer": 16},
    {"id": "246", "problem": "If f(f(f(x))) = x and f(1)=2, f(2)=3, find f(3).", "answer": 1},
    {"id": "247", "problem": "If f(x) = f(x-1) + f(x-2) and f(0)=0, f(1)=1, find f(7).", "answer": 13},
    {"id": "248", "problem": "Find $f(5)$ if $f(x+y) = f(x)f(y)$ and $f(1)=2$.", "answer": 32},
    {"id": "249", "problem": "Find the number of edges in a $K_5$ complete graph.", "answer": 10},
    {"id": "250", "problem": "Number of vertices in a soccer ball (truncated icosahedron).", "answer": 60}
]

VERY_DIFFICULT_POOL = [
    {"id": "251", "problem": "Find the remainder when the number of non-negative integer solutions to the equation $a + b + c + d = 5000$ is divided by 100000.", "answer": 42501},
    {"id": "252", "problem": "A sequence is defined by $a_0 = 0$, $a_1 = 1$, and $a_n = 5a_{n-1} - 6a_{n-2}$ for $n \\ge 2$. Find the remainder when $a_{10^9}$ is divided by 10000.", "answer": 625},
    {"id": "253", "problem": "Find the number of positive integer divisors of $10^{99}$ that are perfect squares.", "answer": 2500},
    {"id": "254", "problem": "Find the number of integers $n \\le 1000$ such that $n$ is divisible by \\phi(n).", "answer": 34},
    {"id": "255", "problem": "Find the remainder when the sum $1! + 2! + 3! + \\dots + 10^6!$ is divided by 1000.", "answer": 313},
    {"id": "256", "problem": "A fair coin is flipped until three consecutive heads appear. Find the expected number of flips.", "answer": 14},
    {"id": "257", "problem": "Find the smallest positive integer $n$ such that $n^2 + (n+1)^2$ is a perfect square and $n > 100$.", "answer": 119},
    {"id": "258", "problem": "Find the number of ordered pairs of integers $(x,y)$ such that $x^2 + y^2 = 625$.", "answer": 20},
    {"id": "259", "problem": "Find the sum of all digits in the integer result of $2^{1000}$.", "answer": 1366},
    {"id": "260", "problem": "Find the remainder when \\binom{2000}{1000} is divided by 7.", "answer": 0},
    {"id": "261", "problem": "The sum of the first $n$ terms of a sequence is $S_n = n(n+1)(n+2)$. Find the value of $a_{100}$.", "answer": 30300},
    {"id": "262", "problem": "Find the last two digits of $7^{7^7}$.", "answer": 43},
    {"id": "263", "problem": "Find the smallest $n > 0$ such that $n^2$ ends in the digits 444.", "answer": 38},
    {"id": "264", "problem": "How many ways can 3 red, 3 blue, and 3 green balls be arranged in a row such that no two adjacent balls are the same color?", "answer": 174},
    {"id": "265", "problem": "Find the remainder when $1! + 2! + \\dots + 100!$ is divided by 15.", "answer": 3},
    {"id": "266", "problem": "Find the number of ordered triples $(x,y,z)$ of positive integers such that $x+y+z=20$ and they form the side lengths of a non-degenerate triangle.", "answer": 36},
    {"id": "267", "problem": "Find the sum of the squares of the first 20 positive integers.", "answer": 2870},
    {"id": "268", "problem": "Find the number of trailing zeros in $2024!$.", "answer": 503},
    {"id": "269", "problem": "Find the sum of all $n$ such that $2^n + 1$ is a Fermat prime and $n \\le 100$.", "answer": 31},
    {"id": "270", "problem": "Find the smallest positive integer $n$ such that \\sum_{i=1}^n i > 1,000,000.", "answer": 1414},
    {"id": "271", "problem": "If $x+y+z=1$ and $1/x + 1/y + 1/z = 0$, find $x^2 + y^2 + z^2$.", "answer": 1},
    {"id": "272", "problem": "Find the remainder when $2^{2^{10}}$ is divided by 17.", "answer": 1},
    {"id": "273", "problem": "Find the number of divisors of $120!$ that are prime.", "answer": 30},
    {"id": "274", "problem": "Find the value of $111,111,111^2$ and sum its digits.", "answer": 81},
    {"id": "275", "problem": "How many ways can a 2x10 grid be tiled with 1x2 dominoes?", "answer": 89},
    {"id": "276", "problem": "Find the smallest $n$ such that $n!$ has at least 290 trailing zeros.", "answer": 1170},
    {"id": "277", "problem": "Find the remainder when the number of subsets of \\{1, 2, \\dots, 20\\} that contain at least one prime number is divided by 100000.", "answer": 44480},
    {"id": "278", "problem": "Find the number of 4x4 binary matrices with exactly one 1 in each row and column.", "answer": 24},
    {"id": "279", "problem": "Find the smallest $n > 1$ such that $n$ is both a perfect square and a perfect cube.", "answer": 64},
    {"id": "280", "problem": "Find the sum of the digits of the multiplicative inverse of $7 \\pmod{100}$.", "answer": 7},
    {"id": "281", "problem": "Find the sum of all positive integer divisors of 1000.", "answer": 2340},
    {"id": "282", "problem": "Find the remainder when $2023^{2024}$ is divided by 1000.", "answer": 41},
    {"id": "283", "problem": "Find the value of $L_{10}$ in the Lucas sequence $L_0=2, L_1=1, L_n = L_{n-1} + L_{n-2}$.", "answer": 123},
    {"id": "284", "problem": "Find the number of paths from (0,0) to (4,4) moving only Right or Up that never cross the line $y=x$.", "answer": 14},
    {"id": "285", "problem": "Find the number of three-digit numbers $abc$ such that $a+b+c = 10$.", "answer": 54},
    {"id": "286", "problem": "Find the largest $k$ such that $3^k$ divides $100!$.", "answer": 48},
    {"id": "287", "problem": "Find the sum of the first 10 terms of the sequence $1, 2, 2, 3, 3, 3, 4, 4, 4, 4$.", "answer": 30},
    {"id": "288", "problem": "Find the number of integers $n \\le 1000$ such that $n^2 + 1$ is divisible by 5.", "answer": 400},
    {"id": "289", "problem": "Find the units digit of $3^{4024} + 4^{3024} \\pmod{10}$.", "answer": 7},
    {"id": "290", "problem": "Find the number of ways to choose 3 squares on a 4x4 chessboard such that no two are in the same row or column.", "answer": 96},
    {"id": "291", "problem": "Find the largest prime factor of $3^{10} + 1$.", "answer": 1181},
    {"id": "292", "problem": "Find the sum of the digits of $(10^{15} + 1)^2$.", "answer": 4},
    {"id": "293", "problem": "Find the number of ways to arrange the letters in 'AIMO' such that no letter is in its original position.", "answer": 9},
    {"id": "294", "problem": "Find the area of a regular hexagon with side length 4.", "answer": 42},
    {"id": "295", "problem": "Find the number of non-congruent triangles with integer sides and perimeter 15.", "answer": 7},
    {"id": "296", "problem": "Find the smallest $n$ such that $n/2$ is a square and $n/3$ is a cube.", "answer": 648},
    {"id": "297", "problem": "Find the remainder when $11^{121}$ is divided by 100.", "answer": 11},
    {"id": "298", "problem": "Find the number of divisors of $10^{10}$.", "answer": 121},
    {"id": "299", "problem": "Find the units digit of $7^{7^7} \\pmod{10}$.", "answer": 3},
    {"id": "300", "problem": "Find the value of $x$ such that \\sqrt{x + \\sqrt{x + \\sqrt{x + \\dots}}} = 1 + \\sqrt{x}.", "answer": 0}
]

# --- 🎯 Selection Logic ---

def select_problems():
    """
    Returns a list of problems based on user input or defaults.
    """
    global SET_SIZE
    test_problems = []
    
    # Detect if we are in an interactive environment
    is_interactive = sys.stdin.isatty() or os.environ.get("AIMO_INTERACTIVE") == "1"
    is_direct_run = (__name__ == "__main__")

    if not is_direct_run and not is_interactive:
        # Default values for when imported by lvties9.py in non-interactive mode (e.g. Kaggle)
        SET_SIZE = 50 
        n_easy, n_med, n_hard, n_vhard = 10, 10, 15, 15
        test_problems = (
            random.sample(EASY_POOL, min(n_easy, len(EASY_POOL))) +
            random.sample(MEDIUM_POOL, min(n_med, len(MEDIUM_POOL))) +
            random.sample(HARD_POOL, min(n_hard, len(HARD_POOL))) +
            random.sample(VERY_DIFFICULT_POOL, min(n_vhard, len(VERY_DIFFICULT_POOL)))
        )
        random.shuffle(test_problems)
        return test_problems

    # Interactive Selection
    try:
        print("\n" + "="*50)
        print("📊 --- STRATIFIED BENCHMARK SELECTOR ---")
        print("="*50)
        print(" [0]  Exit Selection")
        print(" [N]  Standard Mixed: 3, 5, 10, 20, 30, 40, 50")
        print("\n [Pool Shortcuts]:")
        print("  - 1s: Easy Pool (1, 11, 111, 1111...)")
        print("  - 2s: Medium Pool (2, 22, 222, 2222...)")
        print("  - 8s: Hard Pool (8, 88, 888, 8888...)")
        print("  - 9s: Very Difficult Pool (9, 99, 999, 9999...)")
        print("\n [Specific ID]:")
        print("  - i[number] (e.g. i1, i10, i250)")
        print("="*50)
        
        user_input = input("Selection: ").strip()
        if not user_input or user_input == '0':
            return []

        vhard_shortcuts = {"9": 1, "99": 2, "999": 3, "9999": 4, "99999": 5}
        easy_shortcuts = {"1": 1, "11": 2, "111": 3, "1111": 4, "11111": 5}
        medium_shortcuts = {"2": 1, "22": 2, "222": 3, "2222": 4, "22222": 5}
        hard_shortcuts = {"8": 1, "88": 2, "888": 3, "8888": 4, "88888": 5}
        
        if user_input.lower().startswith('i') and len(user_input) > 1:
            id_str = user_input.lower().replace('i', '').strip()
            id_formatted = id_str.zfill(2) if id_str.isdigit() else id_str
            all_pools = EASY_POOL + MEDIUM_POOL + HARD_POOL + VERY_DIFFICULT_POOL
            matched_problem = [p for p in all_pools if p['id'] == id_formatted or p['id'].lower() == id_str.lower()]
            if matched_problem:
                return matched_problem[:1]
        
        elif user_input in vhard_shortcuts:
            return random.sample(VERY_DIFFICULT_POOL, min(vhard_shortcuts[user_input], len(VERY_DIFFICULT_POOL)))
        elif user_input in easy_shortcuts:
            return random.sample(EASY_POOL, min(easy_shortcuts[user_input], len(EASY_POOL)))
        elif user_input in medium_shortcuts:
            return random.sample(MEDIUM_POOL, min(medium_shortcuts[user_input], len(MEDIUM_POOL)))
        elif user_input in hard_shortcuts:
            return random.sample(HARD_POOL, min(hard_shortcuts[user_input], len(HARD_POOL)))
        elif user_input.isdigit():
            size = int(user_input)
            if size == 3: n_easy, n_med, n_hard, n_vhard = 1, 1, 0, 1
            elif size == 5: n_easy, n_med, n_hard, n_vhard = 1, 1, 1, 2
            elif size == 10: n_easy, n_med, n_hard, n_vhard = 2, 3, 2, 3
            else:
                n_easy = max(1, int(size * 0.2))
                n_med = max(1, int(size * 0.2))
                n_hard = max(1, int(size * 0.3))
                n_vhard = size - n_easy - n_med - n_hard
            
            res = (
                random.sample(EASY_POOL, min(n_easy, len(EASY_POOL))) +
                random.sample(MEDIUM_POOL, min(n_med, len(MEDIUM_POOL))) +
                random.sample(HARD_POOL, min(n_hard, len(HARD_POOL))) +
                random.sample(VERY_DIFFICULT_POOL, min(n_vhard, len(VERY_DIFFICULT_POOL)))
            )
            random.shuffle(res)
            return res

    except Exception as e:
        print(f"⚠️ Error during selection: {e}")
    
    return random.sample(EASY_POOL, 5)

# Initialize for import compatibility
TEST_PROBLEMS = select_problems()

# --- 🚀 Standalone Preview & Loop ---
if __name__ == "__main__":
    while True:
        if TEST_PROBLEMS:
            print(f"\n✅ Working with {len(TEST_PROBLEMS)} problems.")
            for i, prob in enumerate(TEST_PROBLEMS):
                print(f" [{i+1}/{len(TEST_PROBLEMS)}] ID: {prob['id']} | Problem: {prob['problem'][:80]}...")
        else:
            print("\n🚪 Selection cleared.")

        print("\n" + "-"*40)
        choice = input("🔄 Choose another set? (Enter to re-select, or 'q' to quit): ").strip().lower()
        if choice == 'q':
            break
        
        TEST_PROBLEMS = select_problems()


Overwriting test.py
